# 🛡️ PeDaS 2026: Deteksi Phishing Domain (.id)
### **Pesta Data Nasional (PeDaS 2026) | APTIKOM Fest 2026 x PANDI**

**Topik Kasus:** *Deteksi Phishing: Untuk Internet Indonesia yang Aman*  
**Mitra Industri:** PANDI (Pengelola Nama Domain Internet Indonesia)  

---

### **Tujuan Notebook:**
1. Menyiapkan alur kerja (*pipeline*) end-to-end ekstraksi fitur phishing khusus ekosistem domain `.id`.
2. Mengekstraksi fitur leksikal, statistik karakter, Shannon Entropy, serta **Brand Impersonation / Combosquatting** terhadap entitas perbankan dan fintech Indonesia.
3. Melakukan validasi model baseline menggunakan **Stratified 5-Fold Cross Validation** bebas kebocoran data (*data-leakage free*).
4. Memastikan kesiapan dan reproduktibilitas kode untuk lingkungan **Google Colab** dan **GitHub** sesuai regulasi resmi PeDaS 2026.

## 1. Setup Lingkungan & Dependensi
Sel di bawah ini secara otomatis mendeteksi apakah kode berjalan di Google Colab atau lingkungan lokal.

In [ ]:
import sys
import os
from pathlib import Path

# Deteksi Lingkungan Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("💻 Running in Google Colab environment.")
    # Clone repository jika belum ada
    if not os.path.exists("PEDAS-2026"):
        # Sesuaikan URL repo Anda saat pengumpulan
        print("Cloning repository...")
        # !git clone https://github.com/<username>/PEDAS-2026.git
        # %cd PEDAS-2026
    if os.path.exists("requirements.txt"):
        !pip install -q -r requirements.txt
except ImportError:
    IN_COLAB = False
    print("🖥️ Running in Local Environment.")

# Pastikan root workspace ada dalam sys.path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.parent))

print(f"Workspace Root: {PROJECT_ROOT}")

## 2. Import Libraries & Inisialisasi Konfigurasi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Set style visualisasi
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# Import modul proyek dari src/
from src.features.extractor import PhishingFeatureExtractor
from src.models.baseline import BaselineModelTrainer
from src.models.metrics import calculate_classification_metrics
from src.utils.config import RANDOM_STATE, BENCHMARK_DATA_DIR, BRANDS_CONFIG_PATH

print(f"✓ Modul proyek berhasil dimuat. Random State: {RANDOM_STATE}")

## 3. Eksplorasi Data Benchmark Domain (.id)
Memuat dataset benchmark awal yang merepresentasikan kasus nyata phishing dan legitimate pada ekosistem domain `.id` (termasuk contoh studi kasus dari materi sosialisasi PANDI).

In [ ]:
data_path = BENCHMARK_DATA_DIR / "sample_phishing_id.csv"
df = pd.read_csv(data_path)

print(f"Dimensi Data: {df.shape[0]} baris x {df.shape[1]} kolom\n")
display(df.head(10))

print("\nDistribusi Label Target:")
print(df["label"].value_counts(normalize=True).rename({0: "Legitimate (0)", 1: "Phishing (1)"}))

print("\nDistribusi Kategori Target:")
print(df["category"].value_counts())

**Analisis Awal Data:**
1. Dataset benchmark awal mencakup proporsi seimbang antara situs phishing (label = 1) dan legitimate (label = 0).
2. Kategori target didominasi oleh sektor yang paling sering diserang di Indonesia: **perbankan (banking)**, **fintech/e-wallet**, **e-commerce**, dan **layanan publik/bansos**.
3. Pola ancaman terlihat jelas: pelaku menggunakan domain murah seperti `.my.id`, `.web.id`, `.biz.id` yang dikombinasikan dengan teknik combosquatting (menyisipkan nama brand resmi seperti BCA, Mandiri, BRI, DANA).

## 4. Ekstraksi Fitur Terpadu (Feature Extraction Pipeline)
Menjalankan `PhishingFeatureExtractor` untuk mengekstrak seluruh fitur leksikal, statistik keacakan (Shannon Entropy), rasio karakter khusus, serta indikator brand spoofing Indonesia.

In [ ]:
extractor = PhishingFeatureExtractor(include_dns=False, include_whois=False)
features_df = extractor.transform(df, url_col="url", show_progress=True)

print(f"Total Fitur Berhasil Diekstrak: {features_df.shape[1]} fitur")
display(features_df.head())

# Pastikan tidak ada missing values tak tertangani
assert not features_df.isna().any().any(), "Terdapat nilai NaN dalam feature matrix!"
print("✓ Seluruh fitur numerik valid dan bebas NaN.")

## 5. Analisis Fitur & Visualisasi Pola Phishing
Mari kita visualisasikan perbedaan karakteristik antara URL Phishing vs Legitimate.

In [ ]:
analysis_df = pd.concat([df[["url", "label", "category"]], features_df], axis=1)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Panjang URL
sns.boxplot(data=analysis_df, x="label", y="url_len", ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Distribusi Panjang URL (url_len)")
axes[0, 0].set_xticklabels(["Legitimate (0)", "Phishing (1)"])

# 2. Shannon Entropy Domain
sns.kdeplot(data=analysis_df, x="domain_entropy", hue="label", fill=True, common_norm=False, ax=axes[0, 1], palette="Set1")
axes[0, 1].set_title("Distribusi Shannon Entropy Domain (domain_entropy)")

# 3. Frekuensi Penggunaan HTTPS
https_ct = pd.crosstab(analysis_df["label"], analysis_df["is_https"], normalize="index") * 100
https_ct.plot(kind="bar", stacked=True, ax=axes[1, 0], colormap="coolwarm", edgecolor="black")
axes[1, 0].set_title("Persentase Penggunaan HTTPS (is_https)")
axes[1, 0].set_xticklabels(["Legitimate (0)", "Phishing (1)"], rotation=0)
axes[1, 0].set_ylabel("Persentase (%)")

# 4. Brand Combosquatting / Impersonation
brand_ct = pd.crosstab(analysis_df["label"], analysis_df["is_unauthorized_brand_domain"], normalize="index") * 100
brand_ct.plot(kind="bar", stacked=True, ax=axes[1, 1], colormap="viridis", edgecolor="black")
axes[1, 1].set_title("Penyalahgunaan Nama Brand Resmi (is_unauthorized_brand_domain)")
axes[1, 1].set_xticklabels(["Legitimate (0)", "Phishing (1)"], rotation=0)
axes[1, 1].set_ylabel("Persentase (%)")

plt.tight_layout()
plt.show()

**Temuan Utama dari Visualisasi:**
1. **Panjang URL & Path**: URL phishing cenderung signifikan lebih panjang karena pelaku menyisipkan kata kunci social engineering (`verifikasi`, `hadiah`, `aktivasi`) ke dalam domain dan path.
2. **Protokol HTTPS**: Situs phishing lokal domain `.id` mayoritas masih menggunakan HTTP biasa atau SSL gratis tanpa sertifikasi EV, sedangkan situs perbankan resmi selalu menggunakan HTTPS.
3. **Unauthorized Brand Domain**: Fitur `is_unauthorized_brand_domain` memiliki daya pisah (*discriminative power*) yang sangat kuat. Domain legitimate yang memuat nama brand (misal `klikbca.com`, `dana.id`) terdaftar resmi, sedangkan domain phishing memanfaatkan nama brand tersebut secara tidak sah pada domain lain.

## 6. Pelatihan Model Baseline (Stratified 5-Fold CV)
Melatih model baseline GBDT (**LightGBM**) menggunakan validasi silang 5-Fold terstratifikasi untuk mengevaluasi performa tanpa kebocoran data.

In [ ]:
X = features_df.copy()
y = df["label"].values

trainer = BaselineModelTrainer(model_type="lightgbm", n_splits=5, random_state=RANDOM_STATE)
overall_metrics, fi_df, oof_probs = trainer.cross_validate(X, y)

print("=" * 45)
print("HASIL EVALUASI BASELINE MODEL (LIGHTGBM)")
print("=" * 45)
for metric_name, val in overall_metrics.items():
    if metric_name != "fold_f1_macros":
        print(f"{metric_name:<20}: {val}")
print("=" * 45)
print(f"Skor F1-Macro per Fold : {overall_metrics['fold_f1_macros']}")

## 7. Analisis Kepentingan Fitur (*Feature Importance*)
Melihat 15 fitur paling berpengaruh dalam pengambilan keputusan model deteksi phishing.

In [ ]:
top_fi = fi_df.head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_fi, x="importance", y="feature", palette="magma")
plt.title("Top 15 Feature Importance - Deteksi Phishing PeDaS 2026")
plt.xlabel("Mean Importance across 5 Folds")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 8. Ringkasan & Langkah Selanjutnya (Warm-Up PeDaS 2026)

### **Hasil yang Dicapai:**
1. **Pipeline Terstruktur**: Modul ekstraksi fitur modular (`src/features/`) dapat memproses URL tunggal maupun batch DataFrame secara cepat dan konsisten.
2. **Fitur Khusus Indonesia**: Kamus brand perbankan/fintech dan indikator combosquatting terbukti memberikan sinyal klasifikasi yang sangat kuat.
3. **Kesesuaian Regulasi Kompetisi**: Seluruh kode ditulis murni dalam **Python**, deterministik dengan `random_state=42`, dan siap disinkronkan ke **GitHub** serta dijalankan via **Google Colab**.

### **Langkah Menyambut Dataset Resmi PANDI (12 September 2026):**
- [ ] Mengintegrasikan kolom data statistik DNS (`dns.id`) dari PANDI menggunakan `DNSFeatureExtractor.extract_from_record_dict()`.
- [ ] Mengintegrasikan data tanggal registrasi WHOIS menggunakan `WHOISFeatureExtractor.extract_from_dict()`.
- [ ] Melakukan tuning hyperparameter (Optuna) dan ensembling (LightGBM + CatBoost + XGBoost) saat submission babak penyisihan dibuka (14 – 25 September 2026).